# 区域并联耦合老化

Canonical workflow for regional parallel-coupled aging. 本 Notebook 合并旧版 `regional_parallel_demo` 的 OCV+R 区域分流、局部堵塞/失效和局部 vs 全局堵塞 sweep，以及 `regional_parallel_314_dfn_demo` / `simulation_regional_lifecycle` 的区域 DFN 等压耦合、恒功率生命周期演化、区域电流分配、孔隙率、负极电位/析锂诊断与导出入口。

所有可变参数只在下一格 `CONFIG` 修改；核心 spec、runner、workflow_id 和输出目录独立于 PSD workflow。

In [ ]:
CONFIG = {
    "cell": "314Ah",
    "run_mode": "smoke",
    "output_name": "区域并联耦合老化",
    "area_fractions": {"A_corner": 0.10, "B_middle": 0.70, "C_top": 0.20},
    "temperature_c": 25.0,
    "aging_t_factor": 1,
    "nominal_capacity_ah": 314.0,
    "nominal_voltage_v": 3.2,
    "reduced_network": {
        "enabled": True,
        "base_ocv_v": 3.30,
        "base_resistance_ohm": 0.00025,
        "current_c_rate": 1.0,
        "voltage_cutoff_v": 2.50,
        "local_blocked_region": "A_corner",
        "sweep_multipliers": [1, 2, 5, 10, 20, 50, 100],
        "cases": [
            {"label": "healthy"},
            {
                "label": "A_blocked_x20",
                "resistance_multipliers": {"A_corner": 20.0, "B_middle": 1.0, "C_top": 1.0},
            },
            {
                "label": "A_failed",
                "resistance_multipliers": {"A_corner": 1000.0, "B_middle": 1.0, "C_top": 1.0},
                "capacity_retention": {"A_corner": 0.0, "B_middle": 1.0, "C_top": 1.0},
                "current_bounds_a": {"A_corner": [0.0, 0.0]},
            },
        ],
    },
    "regional_cases": [
        {"label": "healthy"},
        {
            "label": "A_porosity_30pct",
            "parameter_multipliers_by_region": {
                "A_corner": {
                    "Negative electrode porosity": 0.30,
                    "Separator porosity": 0.30,
                    "Positive electrode porosity": 0.30,
                }
            },
        },
    ],
    "transient_dfn": {
        "enabled": False,
        "current_c_rate": 0.25,
        "duration_s": 600.0,
        "macro_step_s": 60.0,
        "initial_soc": 0.50,
        "relaxation": 0.70,
        "max_iterations": 12,
    },
    "lifecycle": {
        "enabled": False,
        "run_homogeneous_baseline": True,
        "discharge_p_rate": 0.25,
        "cycles": 1,
        "equivalent_cycle_factor": 1.0,
        "rest_s": 600.0,
        "macro_step_s": 300.0,
        "maximum_macro_step_s": 600.0,
        "initial_soc": 1.0,
        "lower_cutoff_v": 2.50,
        "upper_cutoff_v": 3.65,
    },
    "datasets": [
        # Optional DatasetQuery examples for experiment overlays. Set require_unique as needed.
        # {"cell": "314Ah", "temperature_c": 25, "test_type": "区域并联", "kind": "processed", "format": ".csv", "path_contains": "regional", "require_unique": False},
    ],
    "modes": {
        "smoke": {
            "transient_dfn": {"enabled": False},
            "lifecycle": {"enabled": False},
        },
        "study": {
            "transient_dfn": {"enabled": True, "duration_s": 600.0, "macro_step_s": 60.0},
            "lifecycle": {"enabled": True, "cycles": 20, "equivalent_cycle_factor": 50.0},
        },
    },
}

## 环境与配置校验

Notebook 只负责装载 CONFIG、构造 spec 和调用 headless runner。实验输入如需接入，通过 CONFIG.datasets 的 DatasetQuery 走 datasets.json。

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
search_roots = (cwd, *cwd.parents)
WORKSPACE_ROOT = next(root for root in search_roots if (root / "BatteryProject" / "src").exists())
PROJECT_ROOT = WORKSPACE_ROOT / "BatteryProject"
for item in (WORKSPACE_ROOT, PROJECT_ROOT):
    item_text = str(item)
    if item_text not in sys.path:
        sys.path.insert(0, item_text)

import src.workflows.regional_coupled_aging as regional_workflow
importlib.reload(regional_workflow)

RegionalCoupledAgingWorkflowSpec = regional_workflow.RegionalCoupledAgingWorkflowSpec
build_feature_parity_table = regional_workflow.build_feature_parity_table
run_regional_coupled_aging_workflow = regional_workflow.run_regional_coupled_aging_workflow

spec = RegionalCoupledAgingWorkflowSpec.from_mapping(CONFIG)
print("workflow_id:", regional_workflow.WORKFLOW_ID)
print("run_mode:", spec.run_mode)
print("output:", PROJECT_ROOT / "output" / "runs" / regional_workflow.WORKFLOW_ID)
display(pd.DataFrame([spec.to_dict()]))

## Feature parity

保留旧区域并联/局部堵塞/异质性、区域电流分配、生命周期演化、图表、诊断与导出能力并集；重复实现下沉到 src workflow runner。

In [ ]:
feature_parity = build_feature_parity_table()
display(feature_parity)

## 运行 workflow

`smoke` 默认只跑 OCV+R reduced network，用于快速验证区域分流、局部堵塞、失效断开和局部/全局堵塞 sweep。`study` 可启用短时区域 DFN 与生命周期老化；长周期不要在 smoke 下运行。

In [ ]:
result = run_regional_coupled_aging_workflow(
    spec,
    project_root=PROJECT_ROOT,
    workspace_root=WORKSPACE_ROOT,
    run_id=CONFIG.get("run_id"),
)

print("run_dir:", result["context"].run_dir)
display(result["metrics"])

## OCV+R 区域分流与局部堵塞 sweep

这一段对应旧 `regional_parallel_demo`：健康、A 区局部堵塞、A 区失效断开，以及相同阻抗倍率下局部堵塞和全局均匀退化的端电压差异。

In [ ]:
reduced = result["reduced_network"]
case_df = reduced["case_df"]
summary_df = reduced["summary_df"]
sweep_df = reduced["sweep_df"]

display(summary_df)
display(case_df.head(20))

if not case_df.empty:
    pivot = case_df.pivot(index="case", columns="region", values="current_a")
    ax = pivot.plot(kind="bar", figsize=(8.5, 4.2), width=0.72)
    ax.set_ylabel("Current / A")
    ax.set_title("Regional current redistribution")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()

if not sweep_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    axes[0].semilogx(sweep_df["blockage_multiplier"], sweep_df["blocked_current_share_pct"], "o-")
    axes[0].set_xlabel("Blockage multiplier")
    axes[0].set_ylabel("Blocked-region current share / %")
    axes[0].grid(alpha=0.25)
    axes[1].semilogx(sweep_df["blockage_multiplier"], sweep_df["local_terminal_voltage_v"], "o-", label="local")
    axes[1].semilogx(sweep_df["blockage_multiplier"], sweep_df["global_terminal_voltage_v"], "o-", label="global")
    axes[1].axhline(spec.reduced_network.voltage_cutoff_v, color="black", ls="--", lw=1.0)
    axes[1].set_xlabel("Blockage multiplier")
    axes[1].set_ylabel("Terminal voltage / V")
    axes[1].legend()
    axes[1].grid(alpha=0.25)
    plt.tight_layout()

## 区域DFN macro-step 耦合

启用 CONFIG.transient_dfn.enabled 后，runner 调用 `run_regional_dfn_coupling`，保留旧 314Ah notebook 的回滚试算、KCL error、电压 spread、区域电流、C-rate 和孔隙率历史。

In [ ]:
transient = result["transient_dfn"]
steps_df = transient["steps_df"]
iterations_df = transient["iterations_df"]

if steps_df.empty:
    print("transient_dfn 未启用；在 CONFIG.modes.study.transient_dfn 中打开。")
else:
    display(steps_df.head(20))
    display(iterations_df.head(20))
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    for (case, region), group in steps_df.groupby(["case", "region"]):
        axes[0].plot(group["time_s"] / 60.0, group["current_a"], "o-", label=f"{case}-{region}")
    axes[0].set_xlabel("Time / min")
    axes[0].set_ylabel("Current / A")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.25)
    iteration_view = iterations_df.drop_duplicates(["case", "step", "iteration"])
    for (case, step), group in iteration_view.groupby(["case", "step"]):
        axes[1].semilogy(group["iteration"], group["voltage_spread_v"] * 1000.0, "o-", label=f"{case} step {step}")
    axes[1].axhline(1.0, color="black", ls="--", lw=1.0)
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Voltage spread / mV")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.25)
    plt.tight_layout()

## 孔隙率历史

短时 DFN 和生命周期 steps 表均保留负极、隔膜、正极区域平均孔隙率列，用于观察局部堵孔初值和后续老化演化。

In [ ]:
porosity_source = steps_df if not steps_df.empty else result["lifecycle"]["steps_df"]
porosity_columns = ["negative_porosity", "separator_porosity", "positive_porosity"]
if porosity_source.empty or not set(porosity_columns).issubset(porosity_source.columns):
    print("当前 run 没有孔隙率历史；启用 transient_dfn 或 lifecycle 后查看。")
else:
    display(porosity_source[["case", "region", *porosity_columns]].head(20))
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    x_col = "time_s" if "time_s" in porosity_source.columns else "global_time_s"
    for (case, region), group in porosity_source.groupby(["case", "region"]):
        ax.plot(group[x_col] / 60.0, group["negative_porosity"], "o-", label=f"{case}-{region}")
    ax.set_xlabel("Time / min")
    ax.set_ylabel("Negative porosity")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    plt.tight_layout()

## 生命周期演化与负极电位诊断

启用 CONFIG.lifecycle.enabled 后，runner 调用 `run_regional_dfn_power_cycles`，输出区域 step、cycle、degradation 和 iteration 表；同质 baseline 可选，用于与区域异质老化对照。

In [ ]:
lifecycle = result["lifecycle"]
cycle_df = lifecycle["cycles_df"]
lifecycle_steps = lifecycle["steps_df"]
degradation_df = lifecycle["degradation_df"]

if cycle_df.empty:
    print("lifecycle 未启用；在 CONFIG.modes.study.lifecycle 中打开。")
else:
    display(cycle_df)
    display(degradation_df.head(20))
    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
    for case, group in cycle_df.groupby("case"):
        axes[0].plot(group["equivalent_cycle"], group["capacity_retention_pct"], "o-", label=case)
    axes[0].set_xlabel("Equivalent cycle")
    axes[0].set_ylabel("Capacity retention / %")
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    if "negative_surface_potential_difference_min_v" in lifecycle_steps.columns:
        charge = lifecycle_steps[lifecycle_steps["segment"].eq("charge")]
        for (case, region), group in charge.groupby(["case", "region"]):
            axes[1].plot(group["equivalent_cycle"], group["negative_surface_potential_difference_min_v"], "o", label=f"{case}-{region}")
        axes[1].set_xlabel("Equivalent cycle")
        axes[1].set_ylabel("Negative potential min / V")
        axes[1].legend(fontsize=8)
        axes[1].grid(alpha=0.25)
    plt.tight_layout()

## Artifact 链接

所有原始运行、中间表和 Excel 汇总写入 `BatteryProject/output/runs/regional_coupled_aging/<run_id>/`。

In [ ]:
artifact_table = pd.DataFrame(
    [{"name": name, "path": str(path)} for name, path in result["artifact_paths"].items()]
).sort_values("name")
display(artifact_table)
print("artifact_manifest:", result["artifact_paths"].get("artifact_manifest"))